<a href="https://colab.research.google.com/github/harishkulkarni10/ai-research-assistant-rag/blob/dev/notebooks/01_ingestion/arxiv_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pandas pyarrow

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [ ]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

In [ ]:
load_dataset

## Step 1 & 2: Load the dataset

Dataset: ccdv/arxiv-summarization  
Contains ~215k arXiv papers with clean full article text + abstract. No PDF processing needed.

# Loading the arXiv dataset

In [ ]:
# Load the dataset (it has train/validation/test splits — we'll use train for corpus)
ds = load_dataset("ccdv/arxiv-summarization", split="train")

print(f"Dataset loaded!")
print(f"Total papers: {len(ds):,}")
print("\nSchema:")
print(ds.features)
print("\nSample entry keys:", list(ds[0].keys()))

## Step 3: Filter to AI/ML papers

Simple keyword filtering on title + abstract (lowercase).
Keywords chosen to capture core AI/ML topics without too much noise.

In [ ]:
# Define AI/ML keywords (we can expand this as the project goes ahead)

ai_ml_keywords = [
    "machine learning", "deep learning", "neural network", "transformer",
    "large language model", "llm", "gpt", "bert", "artificial intelligence",
    "computer vision", "natural language processing", "nlp", "reinforcement learning",
    "generative ai", "diffusion model", "rag", "retrieval augmented"
]

# Combining title and abstract for searching
def contains_ai_ml(example):
  text = (example['article'] + ' ' + example['abstract']).lower()
  return any(keyword in text for keyword in ai_ml_keywords)

# Filter the dataset
filtered_ds = ds.filter(contains_ai_ml)

print(f"Filtered papers: {len(filtered_ds):,}")
print(f"Reduction: {len(ds) - len(filtered_ds):,} papers removed")

## Step 4: Limit dataset size (baseline)

We'll take 1,000 papers for fast prototyping.
Shuffled with fixed seed for reproducibility.

In [ ]:
np.random.seed(42)
limited_ds = filtered_ds.shuffle(seed=42).select(range(1000))

print(f"Final baseline corpus size: {len(limited_ds):,} papers")

## Step 5: Create clean DataFrame corpus

Columns:
- paper_id: index (temporary unique ID)
- title: not directly available → we'll use first 200 chars of abstract as proxy
- abstract: full abstract
- full_text: article body

In [ ]:
df = pd.DataFrame(limited_ds)

# paper_id
df['paper_id'] = range(len(df))

df['title'] = df['abstract'].str.slice(0, 200) + "..."

df = df.rename(columns={'article': 'full_text'})

# Reorder columns
df = df[['paper_id', 'title', 'abstract', 'full_text']]

print("Corpus DataFrame created!")
df.head()

# Save artifact

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive_path = Path("/content/drive/MyDrive/rag-arxiv-qa/data/processed")
drive_path.mkdir(parents=True, exist_ok=True)

output_path = drive_path / "arxiv_ai_ml_corpus.parquet"

df.to_parquet(output_path, index=False)

print(f"Saved to Google Drive: {output_path}")

# Validation

In [ ]:
print(f"Final number of papers: {len(df):,}")

# Average lengths
df['abstract_len'] = df['abstract'].str.len()
df['full_text_len'] = df['full_text'].str.len()

print(f"\nAverage abstract length: {df['abstract_len'].mean():.0f} chars")
print(f"Average full text length: {df['full_text_len'].mean():.0f} chars")

# Sample inspection
print("\nSample 1:")
print("Title proxy:", df.iloc[0]['title'])
print("Abstract preview:", df.iloc[0]['abstract'][:500])
print("Full text preview:", df.iloc[0]['full_text'][:500])

print("\nSample 2:")
print("Title proxy:", df.iloc[1]['title'])
print("Abstract preview:", df.iloc[1]['abstract'][:500])